# Chapter 10 · Evaluation and Limits

Four questions to the GPT of chapter 7: does it memorize or generalize? Does it work out of its element? Does it tell the truth, or does it only write well? What else has it learned from its data? PyTorch on the CPU. The whole notebook takes about four minutes.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Constants and model

In Colab, the cell downloads the model, the novel and the text of chapters 1 to 4 if they are not there.

In [1]:
"""Chapter 10: what a number proves. PyTorch on CPU."""
import math
import re
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

ROOT = Path(".")
MODEL = Path("material/gpt_quixote.pt")
if not MODEL.exists():  # in Colab, download it from the repository
    import urllib.request
    MODEL.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/HernanDiaz/language-models/main/material/gpt_quixote.pt", MODEL)
TEXT = Path("material/quixote.txt")
if not TEXT.exists():  # in Colab, download it from the repository
    import urllib.request
    TEXT.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/HernanDiaz/language-models/main/material/quixote.txt", TEXT)
OTHER_TEXT = Path("material/corpus_chapters_1_4.txt")
if not OTHER_TEXT.exists():  # in Colab, download it from the repository
    import urllib.request
    OTHER_TEXT.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/HernanDiaz/language-models/main/material/corpus_chapters_1_4.txt", OTHER_TEXT)
SEED = 1337
SMALL = dict(context=32, dimension=32, heads=2, layers=2)
SMALL_CHARACTERS = 2000  # text on which the small model is trained
SEQUENCES = 64           # texts of 2000 characters generated for the biases
FACTS = [("Dulcinea del ", "Toboso"), ("Sancho ", "Panza"),
         ("Don Quixote of La ", "Mancha"), ("the curate and the ", "barber"),
         ("the Knight of the Rueful ", "Countenance"),
         ("the bachelor Samson ", "Carrasco"), ("Mambrino’s ", "helmet"),
         ("Cide Hamete ", "Benengeli")]
PAIRS = [("knight", "lady"), ("father", "mother"), ("king", "queen"),
         ("master", "mistress"), ("he", "she"), ("man", "woman")]


# --- The model of chapters 6 to 9 --------------------------------------------

class Head(nn.Module):
    def __init__(self, config):
        super().__init__()
        C, T = config["dimension"], config["context"]
        size = C // config["heads"]
        self.query = nn.Linear(C, size, bias=False)
        self.key = nn.Linear(C, size, bias=False)
        self.value = nn.Linear(C, size, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(T, T)))

    def forward(self, x):
        T = x.shape[1]
        q, k, v = self.query(x), self.key(x), self.value(x)
        scores = q @ k.transpose(1, 2) / math.sqrt(k.shape[-1])
        future = self.mask[:T, :T] == 0
        scores = scores.masked_fill(future, float("-inf"))
        return F.softmax(scores, dim=-1) @ v


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        C = config["dimension"]
        self.heads = nn.ModuleList(
            [Head(config) for _ in range(config["heads"])])
        self.projection = nn.Linear(C, C)
        self.mlp = nn.Sequential(nn.Linear(C, 4 * C), nn.ReLU(),
                                 nn.Linear(4 * C, C))
        self.norm1, self.norm2 = nn.LayerNorm(C), nn.LayerNorm(C)

    def forward(self, x):
        h = self.norm1(x)
        attention = torch.cat([head(h) for head in self.heads], dim=-1)
        x = x + self.projection(attention)
        return x + self.mlp(self.norm2(x))


class GPT(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        C = config["dimension"]
        self.config = config
        self.symbol_embedding = nn.Embedding(vocab_size, C)
        self.position_embedding = nn.Embedding(config["context"], C)
        self.blocks = nn.Sequential(
            *[Block(config) for _ in range(config["layers"])])
        self.norm = nn.LayerNorm(C)
        self.output = nn.Linear(C, vocab_size)

    def forward(self, ids):
        positions = torch.arange(ids.shape[1])
        x = self.symbol_embedding(ids) + self.position_embedding(positions)
        return self.output(self.norm(self.blocks(x)))


def load_model(path=MODEL):
    saved = torch.load(path)
    symbols = saved["symbols"]
    model = GPT(len(symbols), saved["config"])
    model.load_state_dict(saved["model"])
    return model, symbols, {c: i for i, c in enumerate(symbols)}

## 2. Measuring tools

In [2]:
# --- Measuring tools ---------------------------------------------------------

def mean_loss(model, x, y):
    logits = model(x)
    return F.cross_entropy(logits.view(-1, logits.shape[-1]), y.view(-1))


@torch.no_grad()
def text_loss(model, data, group=256):
    """Mean loss on a whole text, piece by piece (chapter 7)."""
    T = model.config["context"]
    n = (len(data) - 1) // T
    x, y = data[:n * T].view(n, T), data[1:n * T + 1].view(n, T)
    total = 0.0
    for i in range(0, n, group):
        total += mean_loss(model, x[i:i + group],
                           y[i:i + group]).item() * len(x[i:i + group])
    return total / n


@torch.no_grad()
def continue_texts(model, texts, n, symbols, to_id, temperature=0,
                   generator=None):
    """Continue several texts of the same length at once."""
    context = model.config["context"]
    ids = torch.tensor([[to_id[c] for c in text] for text in texts])
    for _ in range(n):
        logits = model(ids[:, -context:])[:, -1]
        if temperature == 0:
            next_id = logits.argmax(dim=-1, keepdim=True)
        else:
            probabilities = F.softmax(logits / temperature, dim=-1)
            next_id = torch.multinomial(probabilities, 1,
                                        generator=generator)
        ids = torch.cat([ids, next_id], dim=1)
    return ["".join(symbols[i] for i in row[-n:]) for row in ids.tolist()]

## 3. Memorizing or generalizing?

In [3]:
# --- 1. Memorizing or generalizing? ------------------------------------------

def repeat_pieces(model, text, symbols, to_id, generator, samples=500,
                  length=32):
    """Does the model continue a piece of the text as the original does?"""
    starts = torch.randint(len(text) - 2 * length, (samples,),
                           generator=generator).tolist()
    prompts = [text[i:i + length] for i in starts]
    originals = [text[i + length:i + 2 * length] for i in starts]
    generated = continue_texts(model, prompts, length, symbols, to_id)
    exact = sum(g == o for g, o in zip(generated, originals))
    equal = sum(a == b for g, o in zip(generated, originals)
                for a, b in zip(g, o))
    return exact / samples, equal / (samples * length)


def train_small(data, validation, vocab_size, steps=2000,
                interval=250):
    """A small model trained on very little text."""
    torch.manual_seed(SEED)
    model = GPT(vocab_size, SMALL)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3,
                                  weight_decay=0.0)
    generator = torch.Generator().manual_seed(SEED)
    T, history = SMALL["context"], []
    for step in range(steps + 1):
        if step % interval == 0:
            history.append((step, text_loss(model, data),
                            text_loss(model, validation)))
        if step == steps:
            break
        starts = torch.randint(len(data) - T - 1, (32,), generator=generator)
        x = torch.stack([data[i:i + T] for i in starts])
        y = torch.stack([data[i + 1:i + T + 1] for i in starts])
        loss = mean_loss(model, x, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    return model, history

## 4. Out of its element, facts and biases

In [4]:
# --- 2. Out of its element ---------------------------------------------------

def clean(text, to_id):
    """Remove the characters the model does not know and count them."""
    unknown = [c for c in text if c not in to_id]
    cleaned = "".join(c for c in text if c in to_id)
    return cleaned, len(unknown), sorted(set(unknown))


# --- 3. Facts and 4. biases --------------------------------------------------

@torch.no_grad()
def probability(model, start, continuation, to_id):
    """Probability that the model gives to a whole continuation."""
    ids = torch.tensor([[to_id[c] for c in start + continuation]])
    logits = model(ids)[0, len(start) - 1:-1]
    targets = ids[0, len(start):]
    logarithms = F.log_softmax(logits, dim=-1)[range(len(targets)),
                                               targets]
    return logarithms.sum().exp().item()


def check_facts(model, training, symbols, to_id):
    """Does the model complete each expression as the novel does?"""
    rows = []
    for prompt, answer in FACTS:
        with_prompt = training.count(prompt)
        with_answer = training.count(prompt + answer)
        output = continue_texts(model, [prompt], len(answer), symbols, to_id)[0]
        rows.append((prompt + answer, with_answer, with_prompt, output,
                     probability(model, prompt, answer, to_id),
                     probability(model, prompt, output, to_id)))
    return rows


def count(text, word):
    return len(re.findall(r"\b" + word + r"\b", text.lower()))


def compare_pairs(novel, generated):
    """What share of each pair goes to its first word."""
    rows = []
    for a, b in PAIRS:
        in_novel = (count(novel, a), count(novel, b))
        in_generated = (count(generated, a), count(generated, b))
        rows.append(((a, b), in_novel, in_generated))
    return rows


def share(pair):
    return pair[0] / (pair[0] + pair[1])

## 5. The complete experiment

It takes about four minutes.

In [5]:
# --- The complete experiment -------------------------------------------------

def run():
    torch.set_num_threads(4)
    generator = torch.Generator().manual_seed(SEED)
    model, symbols, to_id = load_model()
    text = TEXT.read_text(encoding="utf-8")
    a, b = int(0.8 * len(text)), int(0.9 * len(text))
    training, validation, test = text[:a], text[a:b], text[b:]
    data = {name: torch.tensor([to_id[c] for c in block]) for name, block
            in (("training", training), ("validation", validation),
                ("test", test))}
    results = {}

    print("1. Memorizing or generalizing?")
    results["losses"] = {name: text_loss(model, d)
                         for name, d in data.items()}
    print("   Losses:", results["losses"])
    for name, block in (("training", training),
                        ("test", test)):
        results["repeat " + name] = repeat_pieces(
            model, block, symbols, to_id, generator)
        print("   Repeats pieces of", name, results["repeat " + name])
    start = "In a village of La Mancha, the name of which I have no"
    print("  ", repr(start + continue_texts(model, [start], 40, symbols,
                                            to_id)[0]))
    first, last = 200000, 200000 + SMALL_CHARACTERS
    small, history = train_small(data["training"][first:last],
                                 data["validation"][:50000],
                                 len(symbols))
    results["small"] = history
    print("   Small model with", SMALL_CHARACTERS, "characters:",
          history[-1])
    for i in (0, 900, 1800):
        prompt = training[first + i:first + i + 32]
        original = training[first + i + 32:first + i + 72]
        output = continue_texts(small, [prompt], 40, symbols, to_id)[0]
        print("   ", repr(prompt), "->", repr(output), "· original:",
              repr(original))
    prompt = test[1000:1032]
    print("    New text:", repr(prompt), "->", repr(continue_texts(
        small, [prompt], 40, symbols, to_id)[0]))

    print("2. Out of its element")
    other, n_unknown, unknown = clean(
        OTHER_TEXT.read_text(encoding="utf-8"), to_id)
    results["outside"] = text_loss(
        model, torch.tensor([to_id[c] for c in other]))
    print("   Loss on chapters 1 to 4 of this book: %.3f" %
          results["outside"], "·", n_unknown, "unknown characters:",
          "".join(unknown))

    print("3. Facts")
    results["facts"] = check_facts(model, training, symbols,
                                   to_id)
    for row in results["facts"]:
        phrase, times, with_prompt, output, p_right, p_output = row
        print("   %-36s %4d of %4d -> %-11s %6.2f%% · its output %6.2f%%" % (
            phrase, times, with_prompt, output, 100 * p_right, 100 * p_output))

    print("4. Biases")
    starts = ["In a village of La Mancha"] * SEQUENCES
    generated = "\n".join(continue_texts(model, starts, 2000, symbols, to_id,
                                         temperature=1.0, generator=generator))
    results["pairs"] = compare_pairs(training, generated)
    for pair, in_novel, in_generated in results["pairs"]:
        print("   %-16s novel %-12s %3.0f%% · generated %-9s %3.0f%%" % (
            "/".join(pair), in_novel, 100 * share(in_novel),
            in_generated, 100 * share(in_generated)))
    return results

In [6]:
results = run()

1. Memorizing or generalizing?
   Losses: {'training': 1.4276260387523527, 'validation': 1.4797014107230966, 'test': 1.4783268501950375}
   Repeats pieces of training (0.0, 0.1280625)
   Repeats pieces of test (0.0, 0.1195625)
   'In a village of La Mancha, the name of which I have not to be the\nsame of the same of the same'
   Small model with 2000 characters: (2000, 0.241753488779068, 7.248456330366538)
    'rue,” said Vivaldo; and as he wa' -> 's about to read another paper\nof those h' · original: 's about to read another paper\nof those h'
    'less Nero to\nlook down from that' -> ' height upon the ruin of his Rome in emb' · original: ' height upon the ruin of his Rome in emb'
    'you to love me; and\nfor the love' -> ' you show me you say, and to ; and to wh' · original: ' you show me you say, and even urge, tha'
    New text: 'for it’s all\nfinished; but next ' -> 'less amazed than those who had never to '
2. Out of its element
   Loss on chapters 1 to 4 of this book: 2.001 ·

## Exercises

1. Explain with an example of your own the difference between memorizing and generalizing. Why do we need a test set to tell them apart?

2. The losses of our model are 1.428 in training and 1.478 in test. What do you conclude? What difference would make you suspect that it memorizes?

3. The small model ends with a loss of 7.248 on new text, worse than the uniform prediction. How is it possible for a trained model to be worse than knowing nothing?

4. Change the constant `SMALL_CHARACTERS` so that the small model trains on 20,000 characters instead of 2000. How do the two final losses change? Does it still memorize?

5. Of the characters in chapters 1 to 4 of this book, 717 are unknown to the model. What percentage of the text is that? Why can we not simply ignore them when we talk about how well the model works with that text?

6. According to the chapter's table, after Sancho the novel writes Panza 126 times out of 606. What percentage is that? Why is it not enough for an expression to appear many times for the model to learn it?

7. Add three more expressions from the novel to `FACTS`, check how many times they appear and what the model writes. Does what we have seen hold? (Each beginning must end in a space and use only characters the model knows; for example, the novel uses the curly apostrophe, ’, not the straight one. To try it you do not need to run the whole program: it is enough to call `check_facts`.)

8. If the model respected the proportion of the novel, with 8.6% of mistresses, what would be the probability of seeing no mistress among 23 cases? (Hint: 0.914^23.) And of seeing no woman among 31 cases, with the novel's 18.3% of women? Which of the two differences does the generated text prove? Then change `SEQUENCES` to 640, ten times more text, and run the program again (it takes about twenty-three minutes). What changes in the chapter's table?

9. Exact accuracy gives a zero to `enicsuad` as the answer to dulcinea reversed. Propose a measure that gives partial credit and compute it for this answer. What advantages and drawbacks does it have?

10. Failure report. Choose a model from the previous chapters, for example the assistant of chapter 9, and write a failure report with at least three types of failure, each with its criterion, its measure, its result and its scope.

## References

Inspired by the video [Deep Dive into LLMs like ChatGPT](https://www.youtube.com/watch?v=7xTGNNLPyMI), by Andrej Karpathy, and by [Extracting Training Data from Large Language Models](https://arxiv.org/abs/2012.07805) (Carlini and colleagues, 2021). Text of *Don Quixote*, by Miguel de Cervantes, in John Ormsby's English translation, in the public domain. The code and the experiments are original.